**Projeto 1 - Reconhecimento de Objetos Usando YOLO**

*MS671 - Introdução ao Aprendizado de Máquinas Profundo*


---
**Aluna:**

*   *Amanda Rehder Cedro: 253507*

In [ ]:
# INSTALAÇÃO E IMPORTAÇÕES
import sys
import os
from IPython.display import Image, display
import glob
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')  # Backend não interativo, útil para salvar figuras sem abrir janela
import matplotlib.pyplot as plt

# Instala a biblioteca ultralytics no ambiente atual
!pip install ultralytics

from ultralytics import YOLO
import torch


# CLONAR REPOSITÓRIO DO GITHUB

# URL do repositório do projeto
REPO_URL = "https://github.com/Amandacedro/Deep-Learning-P1.git"

# Nome da pasta do repositório
REPO_NAME = "Deep-Learning-P1"

# Caminho completo onde o repositório ficará no Colab
FULL_REPO_PATH = os.path.join("/content", REPO_NAME)

# Garante que estamos no diretório principal do Colab
%cd /content

# Remove uma versão antiga do repositório, se já existir
# Isso evita conflitos com arquivos antigos
if os.path.exists(FULL_REPO_PATH):
    print("Removendo versão antiga do repositório...")
    !rm -rf {FULL_REPO_PATH}

# Clona o repositório do GitHub do zero
print(f"Clonando repositório {REPO_NAME}...")
!git clone {REPO_URL}

# Entra na pasta do projeto clonado
%cd {FULL_REPO_PATH}


# CONFIGURAÇÕES DE DIRETÓRIOS E ARQUIVOS

# Define caminhos principais do projeto
BASE_DIR = "/content/Deep-Learning-P1"
IMAGES_DIR = os.path.join(BASE_DIR, "images")   # Pasta com as imagens de entrada
OUTPUT_DIR = os.path.join(BASE_DIR, "output")   # Pasta para salvar resultados
DATA_DIR = os.path.join(BASE_DIR, "data")       # Pasta para arquivos auxiliares

# Cria as pastas caso ainda não existam
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Garante a existência do arquivo coco.names

# Se o arquivo coco.names estiver em /content, move para data/
if os.path.exists("/content/coco.names"):
    !mv /content/coco.names {DATA_DIR}/
    print(f"Moved /content/coco.names to {DATA_DIR}/coco.names")
else:
    # Se não existir, baixa de uma fonte pública confiável
    print(f"'/content/coco.names' not found. Attempting to download coco.names to {DATA_DIR}/...")
    !wget -q -nc https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names -O {DATA_DIR}/coco.names

# Carrega os nomes das classes do COCO

CLASSES_FILE = os.path.join(DATA_DIR, "coco.names")

# Se o arquivo existir, lê os nomes das classes
if os.path.exists(CLASSES_FILE):
    with open(CLASSES_FILE, "r") as f:
        CLASS_NAMES = [line.strip() for line in f.readlines() if line.strip()]
else:
    # Fallback caso o arquivo não esteja disponível
    print(f"[AVISO] Arquivo {CLASSES_FILE} não encontrado. Usando nomes padrão.")
    CLASS_NAMES = [f"Classe {i}" for i in range(80)]

# Define uma cor aleatória para cada classe
np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(len(CLASS_NAMES), 3), dtype=np.uint8)


# 1. MOSTRAR ARQUITETURA DA REDE NEURAL

def mostrar_arquitetura(model):
    """
    Exibe e salva a arquitetura completa do modelo YOLO.

    A função imprime informações gerais do modelo e também percorre
    suas camadas internas para listar o tipo e a quantidade de parâmetros.
    """
    print("=" * 70)
    print("ARQUITETURA DO MODELO YOLO v11")
    print("=" * 70)

    # Obtém informações do modelo
    info = model.info(verbose=False)

    # Nome do modelo, se disponível
    print(f"Modelo: {model.model_name if hasattr(model, 'model_name') else 'YOLOv11'}")

    # Número de camadas
    print(f"Número de camadas: {info[0] if info else 'N/A'}")

    # Número de parâmetros treináveis
    print(f"Número de parâmetros: {info[1]:,.0f}" if info and len(info) > 1 else "")

    # FLOPs estimados
    print(f"GFLOPs: {info[2]:.1f}" if info and len(info) > 2 else "")
    print()

    print("Camadas da rede:")
    print("-" * 70)

    # Tenta acessar a lista de camadas do modelo
    if hasattr(model, 'model') and model.model:
        for i, layer in enumerate(model.model.model):
            layer_type = layer.__class__.__name__
            params = sum(p.numel() for p in layer.parameters())
            print(f"  Camada {i:3d}: {layer_type:<30s} | Parâmetros: {params:>10,}")
    else:
        print("  Detalhes da camada não disponíveis.")

    print("-" * 70)

    # Salva a arquitetura em um arquivo de texto
    arch_file = os.path.join(OUTPUT_DIR, "arquitetura_yolo.txt")
    with open(arch_file, "w", encoding="utf-8") as f:
        f.write("ARQUITETURA DO MODELO YOLO v11\n")
        f.write("=" * 70 + "\n\n")

        # Escreve estatísticas gerais do modelo
        if info:
            f.write(f"Camadas: {info[0] if info else 'N/A'}\n")
            if len(info) > 1:
                f.write(f"Parâmetros: {info[1]:,.0f}\n")
            if len(info) > 2:
                f.write(f"GFLOPs: {info[2]:.1f}\n")

        f.write("\n")

        # Lista as camadas e subcamadas, caso estejam disponíveis
        if hasattr(model, 'model') and model.model:
            for i, layer in enumerate(model.model.model):
                layer_type = layer.__class__.__name__
                params = sum(p.numel() for p in layer.parameters())
                f.write(f"Camada {i:3d}: {layer_type:<30s} | Parâmetros: {params:>10,}\n")

                # Detalha submódulos internos de cada camada
                for name, sublayer in layer.named_children():
                    f.write(f"           └─ {name}: {sublayer.__class__.__name__}\n")
        else:
            f.write("Detalhes da camada não disponíveis.\n")

    print(f"\nArquitetura salva em: {arch_file}")
    return arch_file


# 2. CARREGAMENTO, REDIMENSIONAMENTO E EXIBIÇÃO DE IMAGENS

def carregar_imagens(images_dir, target_size=640):
    """
    Carrega todas as imagens de uma pasta, redimensiona mantendo a proporção
    e exibe uma galeria com as imagens encontradas.

    Args:
        images_dir: diretório com as imagens
        target_size: tamanho máximo da maior dimensão da imagem

    Returns:
        Dicionário com as imagens originais, redimensionadas e os caminhos.
    """
    imagens = {}

    # Lista apenas arquivos com extensões de imagem conhecidas
    arquivos = sorted([
        f for f in os.listdir(images_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ])

    print(f"\n{'=' * 70}")
    print(f"CARREGAMENTO DE IMAGENS ({len(arquivos)} encontradas)")
    print(f"{'=' * 70}")

    # Cria uma grade de 3x4 para exibição das imagens
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes = axes.flatten()

    for idx, arquivo in enumerate(arquivos):
        caminho = os.path.join(images_dir, arquivo)

        # Lê a imagem com OpenCV
        img = cv2.imread(caminho)
        if img is None:
            print(f"  [AVISO] Não foi possível carregar: {arquivo}")
            continue

        # Obtém dimensões originais
        h_orig, w_orig = img.shape[:2]

        # Redimensiona mantendo proporção
        scale = target_size / max(h_orig, w_orig)
        new_w, new_h = int(w_orig * scale), int(h_orig * scale)
        img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        print(f"  {arquivo}: {w_orig}x{h_orig} → {new_w}x{new_h}")

        # Armazena os dados da imagem
        imagens[arquivo] = {
            "original": img,
            "resized": img_resized,
            "path": caminho
        }

        # Mostra a imagem redimensionada na galeria
        if idx < len(axes):
            axes[idx].imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
            axes[idx].set_title(arquivo, fontsize=10)
            axes[idx].axis("off")

    # Desliga os eixos que ficaram vazios
    for i in range(len(arquivos), len(axes)):
        axes[i].axis("off")

    plt.suptitle("Imagens Carregadas e Redimensionadas", fontsize=16, fontweight="bold")
    plt.tight_layout()

    # Salva a galeria em arquivo
    gallery_path = os.path.join(OUTPUT_DIR, "galeria_imagens.png")
    plt.savefig(gallery_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"\nGaleria salva em: {gallery_path}")
    return imagens


# 3. IMPLEMENTAÇÃO MANUAL DE IoU E NMS

def calcular_iou(box1, box2):
    """
    Calcula Intersection over Union (IoU) entre duas bounding boxes.

    Cada box deve estar no formato:
        [x1, y1, x2, y2]

    IoU = área da interseção / área da união
    """
    # Coordenadas da interseção entre as caixas
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    # Calcula largura e altura da região de interseção
    inter_width = max(0, x2_inter - x1_inter)
    inter_height = max(0, y2_inter - y1_inter)
    area_inter = inter_width * inter_height

    # Calcula as áreas individuais das caixas
    area_box1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area_box2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # Área da união
    area_union = area_box1 + area_box2 - area_inter

    # Evita divisão por zero
    if area_union == 0:
        return 0.0

    return area_inter / area_union


def non_max_suppression_manual(boxes, scores, class_ids, iou_threshold=0.5):
    """
    Implementa manualmente o Non-Max Suppression (NMS).

    O algoritmo faz o seguinte:
    1. Ordena as detecções por confiança
    2. Seleciona a detecção com maior score
    3. Remove as demais caixas da mesma classe com IoU maior que o limiar
    4. Repete até acabar as caixas restantes

    O NMS é aplicado separadamente para cada classe.

    Args:
        boxes: array (N, 4) com [x1, y1, x2, y2]
        scores: array (N,) com scores de confiança
        class_ids: array (N,) com IDs das classes
        iou_threshold: limiar de IoU para remover caixas sobrepostas

    Returns:
        Lista com os índices das caixas mantidas
    """
    if len(boxes) == 0:
        return []

    indices_mantidos = []
    classes_unicas = np.unique(class_ids)

    # Aplica NMS de forma independente para cada classe
    for cls in classes_unicas:
        # Seleciona apenas as detecções da classe atual
        mask_cls = (class_ids == cls)
        indices_cls = np.where(mask_cls)[0]

        if len(indices_cls) == 0:
            continue

        boxes_cls = boxes[indices_cls]
        scores_cls = scores[indices_cls]

        # Ordena índices por score decrescente
        ordem = np.argsort(-scores_cls)
        caixas_restantes = list(ordem)

        while len(caixas_restantes) > 0:
            # Pega a caixa mais confiável
            melhor = caixas_restantes[0]
            indices_mantidos.append(indices_cls[melhor])

            # Compara com as outras caixas restantes
            novas_restantes = []
            for j in caixas_restantes[1:]:
                iou = calcular_iou(boxes_cls[melhor], boxes_cls[j])

                # Mantém apenas caixas com IoU abaixo do limiar
                if iou < iou_threshold:
                    novas_restantes.append(j)

            caixas_restantes = novas_restantes

    # Ordena os índices mantidos antes de retornar
    return sorted(indices_mantidos)


# 4. DETECÇÃO DE OBJETOS

def detectar_objetos(model, imagem_path, conf_threshold=0.5, iou_threshold=0.5):
    """
    Executa a detecção com YOLO e depois aplica NMS manual.

    Fluxo:
    - faz o forward pass do modelo
    - coleta todas as detecções brutas
    - filtra pela confiança mínima
    - aplica NMS manual
    - converte IDs de classe em nomes

    Returns:
        Dicionário com boxes, scores, class_ids, class_names e n_raw
    """
    # Executa o modelo com um limiar bem baixo para capturar muitas caixas brutas
    # O NMS do YOLO é desativado na prática com iou=1.0 e depois é feito manualmente
    results = model.predict(
        source=imagem_path,
        conf=0.01,
        iou=1.0,
        verbose=False,
        save=False
    )

    result = results[0]
    detections = result.boxes

    # Se não houver detecções, retorna estrutura vazia
    if detections is None or len(detections) == 0:
        return {"boxes": [], "scores": [], "class_ids": [], "class_names": [], "n_raw": 0}

    # Extrai caixas, confiança e IDs das classes
    boxes_raw = detections.xyxy.cpu().numpy()
    scores_raw = detections.conf.cpu().numpy()
    class_ids_raw = detections.cls.cpu().numpy().astype(int)

    n_raw = len(boxes_raw)

    # Filtra detecções pela confiança mínima informada
    mask_conf = scores_raw >= conf_threshold
    boxes = boxes_raw[mask_conf]
    scores = scores_raw[mask_conf]
    class_ids = class_ids_raw[mask_conf]

    # Se tudo foi filtrado, devolve resultado vazio
    if len(boxes) == 0:
        return {"boxes": [], "scores": [], "class_ids": [], "class_names": [], "n_raw": n_raw}

    # Aplica NMS manual nas detecções restantes
    indices_nms = non_max_suppression_manual(boxes, scores, class_ids, iou_threshold)

    # Seleciona apenas as caixas mantidas pelo NMS
    boxes_final = boxes[indices_nms]
    scores_final = scores[indices_nms]
    class_ids_final = class_ids[indices_nms]

    # Mapeia IDs de classe para nomes do COCO
    nomes_final = []
    for cid in class_ids_final:
        if cid < len(CLASS_NAMES):
            nomes_final.append(CLASS_NAMES[cid])
        else:
            nomes_final.append(f"classe_{cid}")

    return {
        "boxes": boxes_final,
        "scores": scores_final,
        "class_ids": class_ids_final,
        "class_names": nomes_final,
        "n_raw": n_raw
    }


def desenhar_deteccoes(imagem, deteccoes, titulo=""):
    """
    Desenha bounding boxes, rótulos e confiança sobre a imagem.

    Args:
        imagem: imagem original em BGR
        deteccoes: dicionário retornado por detectar_objetos
        titulo: texto opcional exibido no topo da imagem

    Returns:
        Imagem anotada
    """
    img = imagem.copy()

    # Desenha as detecções, se houver
    if deteccoes and len(deteccoes["boxes"]) > 0:
        for i, (box, score, cls_id, nome) in enumerate(zip(
            deteccoes["boxes"], deteccoes["scores"],
            deteccoes["class_ids"], deteccoes["class_names"]
        )):
            # Converte coordenadas para inteiros
            x1, y1, x2, y2 = box.astype(int)

            # Escolhe cor baseada na classe
            cor_rgb = COLORS[cls_id % len(COLORS)]
            cor = (int(cor_rgb[2]), int(cor_rgb[1]), int(cor_rgb[0]))  # RGB -> BGR

            # Desenha o retângulo da caixa
            cv2.rectangle(img, (x1, y1), (x2, y2), cor, 2)

            # Monta o texto do rótulo
            label = f"{nome}: {score:.2f}"

            # Calcula tamanho do texto para criar fundo colorido
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)

            # Fundo do label
            cv2.rectangle(img, (x1, y1 - th - 10), (x1 + tw + 4, y1), cor, -1)

            # Escreve o texto por cima do fundo
            cv2.putText(img, label, (x1 + 2, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    # Adiciona título no canto superior esquerdo
    if titulo:
        cv2.putText(img, titulo, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

    return img

# 5. COMPARAÇÃO DE LIMIARES NMS E IoU
def comparar_limiares(model, imagem_path, nome_img_base):
    """
    Testa diferentes combinações de limiar de confiança e limiar de IoU.

    Para cada combinação:
    - executa a detecção
    - desenha os resultados
    - salva uma matriz comparativa
    - coleta os resultados em uma tabela
    """
    # Valores testados de confiança e IoU
    conf_thresholds = [0.3, 0.5, 0.7]
    iou_thresholds = [0.3, 0.5, 0.7]

    # Lê a imagem original
    img_original = cv2.imread(imagem_path)

    # Cria a figura 3x3 para comparar os limiares
    fig, axes = plt.subplots(
        len(conf_thresholds), len(iou_thresholds),
        figsize=(24, 20)
    )

    resultados_tabela = []

    for i, conf_th in enumerate(conf_thresholds):
        for j, iou_th in enumerate(iou_thresholds):
            # Executa a detecção com os limiares atuais
            det = detectar_objetos(model, imagem_path, conf_th, iou_th)

            # Conta quantos objetos sobraram após NMS
            n_objetos = len(det["boxes"]) if det and isinstance(det["boxes"], np.ndarray) and len(det["boxes"]) > 0 else 0

            # Define título do subplot
            titulo = f"Conf={conf_th} IoU={iou_th} ({n_objetos} obj)"

            # Desenha as detecções, se houver
            img_det = desenhar_deteccoes(img_original, det, titulo) if n_objetos > 0 else img_original.copy()

            # Exibe no subplot correspondente
            axes[i][j].imshow(cv2.cvtColor(img_det, cv2.COLOR_BGR2RGB))
            axes[i][j].set_title(titulo, fontsize=11, fontweight="bold")
            axes[i][j].axis("off")

            # Registra os resultados em uma lista para tabela/resumo
            classes_det = ", ".join(det["class_names"]) if n_objetos > 0 else "Nenhum"
            resultados_tabela.append({
                "conf": conf_th,
                "iou": iou_th,
                "n_objetos": n_objetos,
                "classes": classes_det
            })

    plt.suptitle(f"Comparação de Limiares - {os.path.splitext(nome_img_base)[0]}", fontsize=16, fontweight="bold")
    plt.tight_layout()

    # Salva a imagem da comparação
    comp_path = os.path.join(OUTPUT_DIR, f"comparacao_limiares_{os.path.splitext(nome_img_base)[0]}.png")
    plt.savefig(comp_path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

    return resultados_tabela


# MAIN
def main():
    """
    Função principal do projeto.

    Etapas:
    1. Carrega o modelo YOLOv11
    2. Exibe a arquitetura
    3. Carrega as imagens da pasta
    4. Executa detecção padrão
    5. Compara diferentes limiares
    6. Salva os resultados em arquivos
    7. Exibe os resultados no notebook
    """
    print("=" * 70)
    print("  PROJETO 1 - RECONHECIMENTO DE OBJETOS USANDO YOLO v11")
    print("  Deep Learning - UNICAMP")
    print("=" * 70)

    # Carregamento do modelo
    print("\nCarregando modelo YOLOv11...")
    model = YOLO("yolo11n.pt")
    print("Modelo carregado com sucesso!")

    # 1) Arquitetura da rede
    mostrar_arquitetura(model)

    # 2) Carregamento das imagens
    imagens = carregar_imagens(IMAGES_DIR)

    # 3 e 4) Detecção e comparação de limiares
    print(f"\n{'=' * 70}")
    print("DETECÇÃO DE OBJETOS E COMPARAÇÃO DE LIMIARES")
    print(f"{'=' * 70}")

    resumo_geral = []

    for nome_img, dados in imagens.items():
        print(f"\n--- Processando: {nome_img} ---")

        # Detecção padrão com limites fixos
        det = detectar_objetos(model, dados["path"], conf_threshold=0.5, iou_threshold=0.5)

        # Conta quantas detecções ficaram após o NMS
        n_obj = len(det["boxes"]) if det and isinstance(det["boxes"], np.ndarray) and len(det["boxes"]) > 0 else 0

        # Mostra estatísticas no terminal
        print(f"  Detecções brutas: {det['n_raw']}")
        print(f"  Objetos após NMS manual (conf=0.5, IoU=0.5): {n_obj}")

        if n_obj > 0:
            for cls_name, score in zip(det["class_names"], det["scores"]):
                print(f"    - {cls_name}: {score:.2%}")

        # Salva a imagem com as detecções
        img_det = desenhar_deteccoes(dados["original"], det)
        det_path = os.path.join(OUTPUT_DIR, f"deteccao_{nome_img}")
        cv2.imwrite(det_path, img_det)

        # Executa a comparação de limiares
        tabela = comparar_limiares(model, dados["path"], os.path.splitext(nome_img)[0])

        # Guarda um resumo geral por imagem
        resumo_geral.append({
            "imagem": nome_img,
            "n_objetos": n_obj,
            "classes": ", ".join(det["class_names"]) if n_obj > 0 else "Nenhum",
            "limiares": tabela
        })

    # Salvar resumo textual
    resumo_path = os.path.join(OUTPUT_DIR, "resumo_deteccoes.txt")
    with open(resumo_path, "w", encoding="utf-8") as f:
        f.write("RESUMO GERAL DAS DETECÇÕES - YOLO v11\n")
        f.write("=" * 80 + "\n\n")

        for r in resumo_geral:
            f.write(f"Imagem: {r['imagem']}\n")
            f.write(f"  Objetos detectados (conf=0.5, IoU=0.5): {r['n_objetos']}\n")
            f.write(f"  Classes: {r['classes']}\n")
            f.write(f"\n  Tabela de Limiares:\n")
            f.write(f"  {'Conf':>6} | {'IoU':>6} | {'N.Obj':>6} | Classes Detectadas\n")
            f.write(f"  {'-'*6} | {'-'*6} | {'-'*6} | {'-'*40}\n")

            for t in r["limiares"]:
                f.write(f"  {t['conf']:>6.1f} | {t['iou']:>6.1f} | {t['n_objetos']:>6} | {t['classes']}\n")

            f.write("\n" + "-" * 80 + "\n\n")

    # Exibição dos resultados no notebook

    print(f"\n{'=' * 70}")
    print("EXIBINDO RESULTADOS:")
    print(f"{'=' * 70}")

    # 1. Galeria inicial
    print("\n--- Galeria Inicial ---")
    display(Image(filename=os.path.join(OUTPUT_DIR, "galeria_imagens.png")))

    # 2. Imagens com detecção padrão
    print("\n--- Detecções (Conf=0.5, IoU=0.5) ---")
    det_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "deteccao_*.jpg")))
    for f in det_files:
        print(f"Resultado: {os.path.basename(f)}")
        display(Image(filename=f))

    # 3. Matrizes de comparação de limiares
    print("\n--- Matrizes de Comparação de Limiares ---")
    comp_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "comparacao_limiares_*.png")))
    for f in comp_files:
        print(f"Matriz: {os.path.basename(f)}")
        display(Image(filename=f))

    # 4. Exibe o resumo textual salvo
    print("\n--- Resumo das Detecções ---")
    with open(resumo_path, "r") as f:
        print(f.read())

    print(f"CONCLUÍDO! Resultados salvos em: {OUTPUT_DIR}")
    print(f"  - Arquitetura: arquitetura_yolo.txt")
    print(f"  - Galeria: galeria_imagens.png")
    print(f"  - Detecções: deteccao_*.jpg")
    print(f"  - Comparações: comparacao_limiares_*.png")
    print(f"  - Resumo: resumo_deteccoes.txt")
    print(f"{'=' * 70}")

# EXECUÇÃO DO PROGRAMA

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
/content
Clonando repositório Deep-Learning-P1...
Cloning into 'Deep-Learning-P1'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 79 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 9.85 MiB | 22.93 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/Deep-Learning-P1
'/content/coco.names' not found. Attempting to download coco.names to /content/Deep-Learning-P1/data/...
  PROJETO 1 - RECONHECIMENTO DE OBJETOS USANDO YO